# Phase 2 Training

Train a fresh policy from a curriculum containing 80% archived sub-800 colorings and 20% random colorings. Phase 1 weights are never loaded.

In [ ]:
# Imports and Project Paths

%matplotlib inline

from pathlib import Path
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np

from ramsey import (
    RArchiveConstruction,
    REnvironment,
    REnvironmentConfig,
    RGraph,
    RMixedConstruction,
    RMonochromaticObjective,
    RProblem,
    RRandomConstruction,
    RSQLiteArchive,
    RTabuMemory,
    RTabuMemoryConfig,
)
from ramsey.RPlot import plot_coloring_histogram
from ramsey.nn import (
    RCheckpointSchedule,
    RModelConfig,
    RPPOConfig,
    RPPOTrainer,
    RPairPolicyValueNetwork,
    RRolloutConfig,
    RRolloutReward,
    RTrainingConfig,
    create_numpy_generator,
    create_optimizer,
    load_training_checkpoint,
    resolve_torch_device,
    seed_torch,
)

project_root = Path.cwd().resolve()

if project_root.name == "notebooks":
    project_root = project_root.parent

if not (project_root / "ramsey").is_dir():
    raise RuntimeError(
        "Run this notebook from the RamseyNumber root or notebooks directory."
    )

In [ ]:
# Phase 2 Experiment Configuration

RANDOM_SEED = 202_608_042
N_VERTICES = 43
RUN_NAME = "phase-2-sub-800-curriculum"

TRAINING_ITERATIONS = 250
CHECKPOINT_INTERVAL = 10
ROLLOUT_STEPS = 128

ARCHIVE_SCORE_LIMIT = 799
MINIMUM_ARCHIVE_COLORINGS = 500
ARCHIVE_SEED_PROBABILITY = 0.80
RANDOM_SEED_PROBABILITY = 0.20

EDGE_TABU_TENURE = 20
VISITED_STATE_WINDOW = 2_000

DATABASE_PATH = (
    project_root
    / "data"
    / "ramsey_colorings.sqlite3"
)

CHECKPOINT_DIRECTORY = (
    project_root
    / "checkpoints"
    / "phase_2"
)

# Leave as None to create a fresh Phase 2 network. Only use a
# checkpoint created by this notebook when resuming Phase 2.
RESUME_PHASE_2_CHECKPOINT: Path | None = None
NEW_LEARNING_RATE: float | None = None

In [ ]:
# Runtime, Problem, and Graph

rng = create_numpy_generator(
    RANDOM_SEED
)

seed_torch(
    RANDOM_SEED
)

device = resolve_torch_device()

graph_start = perf_counter()

problem = RProblem.r55(
    n_vertices=N_VERTICES,
)

graph = RGraph(
    problem
)

graph_elapsed = (
    perf_counter()
    - graph_start
)

print("Project root:", project_root)
print("Device:", device)
print("Problem:", problem)
print("Edges:", f"{graph.number_of_edges:,}")
print("Graph construction:", f"{graph_elapsed:.3f} seconds")

In [ ]:
# Model, Rollout, and PPO Configuration

model_config = RModelConfig(
    input_size=3,
    hidden_size=64,
    number_of_layers=4,
    dropout=0.0,
)

rollout_config = RRolloutConfig(
    rollout_steps=ROLLOUT_STEPS,
    discount=0.995,
    gae_lambda=0.95,
    reward_scale=10.0,
    reward_source=RRolloutReward.EXACT_SCORE,
    normalize_advantages=True,
)

ppo_config = RPPOConfig(
    update_epochs=4,
    minibatch_size=16,
    clip_ratio=0.20,
    value_loss_weight=0.00,
    entropy_weight=0.001,
    maximum_gradient_norm=0.50,
    learning_rate=1.0e-4,
    target_kl=0.02,
)

In [ ]:
# Create a Fresh Phase 2 Model or Resume Phase 2

if RESUME_PHASE_2_CHECKPOINT is None:
    network = RPairPolicyValueNetwork(
        graph,
        model_config,
    ).to(device)

    optimizer = create_optimizer(
        network,
        ppo_config,
    )

    start_iteration = 0

    print("Created a fresh Phase 2 network.")
else:
    checkpoint_path = RESUME_PHASE_2_CHECKPOINT.resolve()
    checkpoint_root = CHECKPOINT_DIRECTORY.resolve()

    if checkpoint_root not in checkpoint_path.parents:
        raise ValueError(
            "Phase 2 may only resume from checkpoints/phase_2."
        )

    restored = load_training_checkpoint(
        checkpoint_path,
        graph=graph,
        device=device,
        rng=rng,
        new_learning_rate=NEW_LEARNING_RATE,
    )

    network = restored.network
    optimizer = restored.optimizer
    model_config = restored.model_config
    rollout_config = restored.rollout_config
    ppo_config = restored.ppo_config
    start_iteration = restored.completed_iteration + 1

    print("Restored Phase 2 checkpoint:", checkpoint_path)

print("Starting iteration:", start_iteration)
print("Parameters:", f"{network.trainable_parameter_count:,}")
print("Learning rate:", optimizer.param_groups[0]["lr"])
print("Rollout steps:", rollout_config.rollout_steps)

In [ ]:
# Open and Validate the Phase 2 Curriculum Archive

existing_archive = globals().get("archive")

if existing_archive is not None:
    existing_archive.close()

archive = RSQLiteArchive(
    DATABASE_PATH
)

eligible_archive_count = (
    archive.coloring_count_in_score_range(
        maximum_score=ARCHIVE_SCORE_LIMIT,
        graph=graph,
    )
)

if eligible_archive_count < MINIMUM_ARCHIVE_COLORINGS:
    archive.close()

    raise RuntimeError(
        f"Phase 2 requires at least {MINIMUM_ARCHIVE_COLORINGS} "
        f"distinct sub-800 colorings; found {eligible_archive_count}."
    )

print("Database:", DATABASE_PATH.resolve())
print("Distinct sub-800 colorings:", eligible_archive_count)
print("Database best:", archive.best_score(graph))

In [ ]:
# Mixed Curriculum and Search Environment

archive_construction = RArchiveConstruction(
    archive=archive,
    rng=rng,
    maximum_score=ARCHIVE_SCORE_LIMIT,
)

random_construction = RRandomConstruction(
    rng
)

construction = RMixedConstruction(
    constructions=(
        archive_construction,
        random_construction,
    ),
    probabilities=(
        ARCHIVE_SEED_PROBABILITY,
        RANDOM_SEED_PROBABILITY,
    ),
    rng=rng,
    construction_name="phase-2-mixed-curriculum",
)

objective = RMonochromaticObjective()

memory = RTabuMemory(
    number_of_edges=graph.number_of_edges,
    config=RTabuMemoryConfig(
        edge_tenure=EDGE_TABU_TENURE,
        visited_state_window=VISITED_STATE_WINDOW,
    ),
)

environment = REnvironment(
    graph=graph,
    objective=objective,
    memory=memory,
    config=REnvironmentConfig(
        max_steps=rollout_config.rollout_steps,
        use_aspiration=True,
    ),
)

print("Construction:", construction.name)
print("Archive source:", archive_construction.name)
print("Archive probability:", ARCHIVE_SEED_PROBABILITY)
print("Random probability:", RANDOM_SEED_PROBABILITY)

In [ ]:
# Assemble the Phase 2 Trainer

trainer = RPPOTrainer(
    graph=graph,
    construction=construction,
    environment=environment,
    network=network,
    optimizer=optimizer,
    device=device,
    rng=rng,
    rollout_config=rollout_config,
    ppo_config=ppo_config,
    archive=archive,
)

checkpoint_schedule = RCheckpointSchedule(
    directory=CHECKPOINT_DIRECTORY,
    interval=CHECKPOINT_INTERVAL,
    save_final=True,
)

training_config = RTrainingConfig(
    run_name=RUN_NAME,
    iterations=TRAINING_ITERATIONS,
    start_iteration=start_iteration,
    stop_on_solution=True,
)

print("Iterations:", training_config.iterations)
print("Checkpoint directory:", CHECKPOINT_DIRECTORY.resolve())

In [ ]:
# Source-Aware Progress Reporting

def report_training_iteration(
    iteration_result,
) -> None:
    metrics = iteration_result.metrics

    if metrics is None:
        metric_text = "no parameter update"
    else:
        metric_text = (
            f"policy={metrics.policy_loss:+.4f} | "
            f"value={metrics.value_loss:.4f} | "
            f"entropy={metrics.entropy:.4f} | "
            f"KL={metrics.approximate_kl:.6f} | "
            f"clipped={metrics.clipped_fraction:.4f}"
        )

    checkpoint_text = ""

    if iteration_result.checkpoint_path is not None:
        checkpoint_text = (
            f" | checkpoint={iteration_result.checkpoint_path.name}"
        )

    database_best_text = (
        " NEW DATABASE BEST"
        if iteration_result.new_archive_best
        else ""
    )

    print(
        f"Iteration {iteration_result.iteration:6d} | "
        f"source={iteration_result.construction_name:24s} | "
        f"initial={iteration_result.initial_score:5d} | "
        f"final={iteration_result.final_score:5d} | "
        f"best={iteration_result.best_score:5d} | "
        f"reward={iteration_result.total_scaled_reward:+9.3f} | "
        f"{metric_text}"
        f"{database_best_text}"
        f"{checkpoint_text}"
    )

In [ ]:
# Run Phase 2 Training

training_start = perf_counter()

training_result = trainer.run(
    training_config,
    checkpoint_schedule=checkpoint_schedule,
    observer=report_training_iteration,
)

training_elapsed = (
    perf_counter()
    - training_start
)

print()
print("Completed iterations:", training_result.completed_iterations)
print("Best score:", training_result.best_score)
print("Solved:", training_result.solved)
print("Total time:", f"{training_elapsed:.3f} seconds")
print(
    "Mean time per iteration:",
    f"{training_elapsed / training_result.completed_iterations:.3f} seconds",
)

In [ ]:
# Compare Archive-Seed and Random-Seed Outcomes

for source_name in (
    archive_construction.name,
    random_construction.name,
):
    source_results = [
        item
        for item in training_result.iteration_results
        if item.construction_name == source_name
    ]

    if not source_results:
        print(source_name, "produced no iterations.")
        continue

    source_initial = np.asarray(
        [item.initial_score for item in source_results],
        dtype=np.float64,
    )

    source_best = np.asarray(
        [item.best_score for item in source_results],
        dtype=np.float64,
    )

    print()
    print("Source:", source_name)
    print("Iterations:", len(source_results))
    print("Mean initial:", f"{source_initial.mean():.2f}")
    print("Mean best:", f"{source_best.mean():.2f}")
    print(
        "Mean best reduction:",
        f"{(source_initial - source_best).mean():.2f}",
    )
    print("Source best:", int(source_best.min()))

In [ ]:
# Plot Phase 2 Scores

iteration_numbers = np.asarray(
    [item.iteration for item in training_result.iteration_results]
)

initial_scores = np.asarray(
    [item.initial_score for item in training_result.iteration_results]
)

best_scores = np.asarray(
    [item.best_score for item in training_result.iteration_results]
)

running_best_scores = np.minimum.accumulate(
    best_scores
)

figure, axis = plt.subplots(
    figsize=(11, 5.5),
    constrained_layout=True,
)

axis.plot(
    iteration_numbers,
    initial_scores,
    label="initial",
    alpha=0.55,
)

axis.plot(
    iteration_numbers,
    best_scores,
    label="rollout best",
    alpha=0.85,
)

axis.plot(
    iteration_numbers,
    running_best_scores,
    label="run best",
    linewidth=2.5,
)

axis.axhline(
    ARCHIVE_SCORE_LIMIT,
    color="black",
    linestyle="--",
    alpha=0.5,
    label="Phase 2 entry threshold",
)

axis.set_xlabel("Training iteration")
axis.set_ylabel("Monochromatic K5 score")
axis.set_title(RUN_NAME)
axis.grid(linestyle="--", alpha=0.3)
axis.legend()

plt.show()

In [ ]:
# Best Coloring and Updated Archive

best_iteration = training_result.best_iteration

print("Best iteration:", best_iteration.iteration)
print("Seed source:", best_iteration.construction_name)
print("Initial score:", best_iteration.initial_score)
print("Final score:", best_iteration.final_score)
print("Best score:", best_iteration.best_score)
print("Database best:", archive.best_score(graph))

plot_coloring_histogram(
    best_iteration.best_coloring,
    title=(
        f"Phase 2 best — iteration {best_iteration.iteration:,} — "
        f"score {best_iteration.best_score:,}"
    ),
)

plt.show()

print()
print("Archive leaderboard")

for record in archive.best_colorings(
    limit=10,
    graph=graph,
):
    print(
        f"ID={record.coloring_id:6d} | "
        f"score={record.score:5d} | "
        f"run={record.run_name} | "
        f"iteration={record.iteration:6d} | "
        f"seen={record.times_seen}"
    )

archive.close()